In [14]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import gc
from opacus import PrivacyEngine
from torch.utils.data import random_split


transform = transforms.Compose([
    transforms.ToTensor(),  
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
train_size_private = int(0.96 * len(train_dataset))
train_size_public = len(train_dataset) - train_size_private
train_dataset_public, train_dataset_private = random_split(train_dataset, [train_size_public, train_size_private])
train_loader_public = DataLoader(dataset=train_dataset_public, batch_size=64, shuffle=True)
train_loader_private = DataLoader(dataset=train_dataset_private, batch_size=64, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class mnist(nn.Module):
    def __init__(self):
        super(mnist, self).__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tang2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(120, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tang2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 120)  # Flatten the tensor
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return x

model = mnist()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

privacy_engine = PrivacyEngine()


gc.collect()

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


0

In [16]:
for epoch in range(5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader_public:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/10]')
    print("Training loss", (running_loss / total))  
    print("Training accuracy", (correct *100/ total),'%')
    gc.collect()


    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss / total))
    print("Testing accuracy", (correct*100 / total),'%')

    gc.collect()

Epoch [1/10]
Training loss 2.2954931386311848
Training accuracy 19.083333333333332 %
Testing loss 0.5509183532714844
Testing accuracy 25.81 %
Epoch [2/10]
Training loss 2.245346711476644
Training accuracy 33.208333333333336 %
Testing loss 0.5388832107543945
Testing accuracy 51.49 %
Epoch [3/10]
Training loss 1.8766823593775432
Training accuracy 60.625 %
Testing loss 0.45040376625061035
Testing accuracy 68.53 %
Epoch [4/10]
Training loss 0.9991514110565185
Training accuracy 74.41666666666667 %
Testing loss 0.23979633865356445
Testing accuracy 80.72 %
Epoch [5/10]
Training loss 0.6128986140092214
Training accuracy 83.0 %
Testing loss 0.14709566736221313
Testing accuracy 84.71 %


In [17]:
model.train()
model, optimizer, train_loader_private = privacy_engine.make_private_with_epsilon(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader_private,
    epochs=10,
    target_epsilon=3,
    target_delta=1e-5,
    max_grad_norm=0.1,
)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader_private:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/10]')
    print("Training loss", (running_loss / total))  
    print("Training accuracy", (correct *100/ total),'%')
    gc.collect()


    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss / total))
    print("Testing accuracy", (correct*100 / total),'%')

    gc.collect()

Epoch [1/10]
Training loss 0.5569567180863354
Training accuracy 82.94788131906378 %
Testing loss 3.2005517804831265
Testing accuracy 83.53 %
Epoch [2/10]
Training loss 0.5438717820804319
Training accuracy 82.87452207160236 %
Testing loss 3.129438234090805
Testing accuracy 84.16 %
Epoch [3/10]
Training loss 0.5200880385149416
Training accuracy 83.96272257124015 %
Testing loss 2.996851295530796
Testing accuracy 85.26 %
Epoch [4/10]
Training loss 0.5096741142878704
Training accuracy 84.85720687337343 %
Testing loss 2.9571801785096525
Testing accuracy 86.17 %
Epoch [5/10]
Training loss 0.4866662553851452
Training accuracy 86.07636603028308 %
Testing loss 2.809134959334135
Testing accuracy 87.09 %
Epoch [6/10]
Training loss 0.47237262008831227
Training accuracy 87.01007065001468 %
Testing loss 2.7346123349532485
Testing accuracy 87.69 %
Epoch [7/10]
Training loss 0.4733996425219367
Training accuracy 87.57561575927755 %
Testing loss 2.7311845576018094
Testing accuracy 88.2 %
Epoch [8/10]
Tra

In [18]:
model.eval()
correct = 0
total = 0
eval_loss = 0.0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        eval_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test Loss: {eval_loss / total:.4f}, Accuracy: {100 * correct / total:.2f}%")

Test Loss: 0.4304, Accuracy: 89.19%
